# Mục lục

## Cài đặt thư viện

In [1]:
!pip install git+https://github.com/thinh-vu/vnstock

  Cloning https://github.com/thinh-vu/vnstock to /tmp/pip-req-build-qxexti7w
  Running command git clone --filter=blob:none --quiet https://github.com/thinh-vu/vnstock /tmp/pip-req-build-qxexti7w
  Resolved https://github.com/thinh-vu/vnstock to commit 7bc8ec5eeda5c107995975d985306405ccbc512e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Nạp thư viện lần đầu

In [2]:
from vnstock import Listing, Company, Finance, Trading, Screener

Non-200 response fetching content: 502



## 👋 Chào mừng bạn đến với Vnstock!
Cảm ơn bạn đã sử dụng package phân tích chứng khoán #1 tại Việt Nam
* Tài liệu: [Sổ tay hướng dẫn](https://vnstocks.com/docs)
* Cộng đồng: [Nhóm Facebook](https://facebook.com/groups/vnstock.official)
Khám phá các tính năng mới nhất và tham gia cộng đồng để nhận hỗ trợ.
                

# II. Truy xuất dữ liệu - CK Việt Nam

## Listing: Liệt kê danh sách chứng khoán

In [ ]:
from vnstock import Listing
listing = Listing()

### Danh sách cổ phiếu theo nhóm

In [ ]:
# Liệt kê tất cả mã chứng khoán theo nhóm phân loại. Ví dụ HOSE, VN30, VNMidCap, VNSmallCap, VNAllShare, VN100, ETF, HNX, HNX30, HNXCon, HNXFin, HNXLCap, HNXMSCap, HNXMan, UPCOM, FU_INDEX (mã chỉ số hợp đồng tương lai), CW (chứng quyền)
listing.symbols_by_group('VN30')

## Quote: Giá chứng khoán

In [ ]:
from vnstock import Quote
import os
quote = Quote(symbol='VN30', source='TCBS')

#### Giá lịch sử

##### Cổ phiếu

In [ ]:
#ACB
from vnstock import Quote
quote = Quote(symbol='ACB', source='TCBS')

df = quote.history(symbol='', start='2020-01-01', end='2025-11-10', interval='1D')
df.to_excel('ACB_2020_2025.xlsx', index=False)
print("Đã lưu file thành công!")

In [ ]:
# VN30 2025
from vnstock import Quote
import os, time, json, re, sys
from tqdm import tqdm
import pandas as pd

# ---- Cấu hình ----
save_folder = os.path.join("docs", "Data_VN30")
os.makedirs(save_folder, exist_ok=True)

progress_file = os.path.join(save_folder, "progress.json")
start_date = '2020-01-01'
end_date   = '2025-11-10'
source     = 'TCBS'   # có thể đổi thành 'VCI' nếu muốn thử nguồn khác
interval   = '1D'

vn30_symbols = [
    'ACB', 'BCM', 'BID', 'CTG', 'DGC', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG',
    'LPB', 'MBB', 'MSN', 'MWG', 'PLX', 'SAB', 'SHB', 'SSB', 'SSI', 'STB',
    'TCB', 'TPB', 'VCB', 'VHM', 'VIB', 'VIC', 'VJC', 'VNM', 'VPB', 'VRE'
]

# ---- Helpers ----
def load_progress(fn):
    if os.path.exists(fn):
        try:
            return json.load(open(fn, 'r'))
        except:
            return {}
    return {}

def save_progress(fn, data):
    with open(fn, 'w') as f:
        json.dump(data, f, indent=2)

def parse_wait_seconds_from_msg(msg):
    if not msg:
        return None
    # tìm pattern tiếng việt "xx giây"
    m = re.search(r'(\d+)\s*giây', msg, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    # tìm pattern tiếng anh "xx seconds" hoặc "retry after xx"
    m2 = re.search(r'(?:(\d+)\s*seconds)|(?:retry after[: ]?(\d+))', msg, flags=re.IGNORECASE)
    if m2:
        for g in m2.groups():
            if g and g.isdigit():
                return int(g)
    return None

# ---- Hàm tải với bắt SystemExit ----
def fetch_symbol_to_file(symbol,
                         start=start_date,
                         end=end_date,
                         source=source,
                         interval=interval,
                         out_folder=save_folder,
                         max_attempts=8,
                         initial_backoff=2.0):
    out_path = os.path.join(out_folder, f"{symbol}_{start.replace('-','')}_{end.replace('-','')}.xlsx")
    if os.path.exists(out_path):
        return True, out_path  # đã có, bỏ qua

    attempt = 0
    backoff = initial_backoff

    while attempt < max_attempts:
        attempt += 1
        try:
            # Thực hiện call
            quote = Quote(symbol=symbol, source=source)
            df = quote.history(symbol=symbol, start=start, end=end, interval=interval)

            # Nếu trả về None hoặc empty -> coi như valid nhưng báo warning
            if df is None or df.empty:
                print(f"⚠️ {symbol}: no data returned. Skipping save.")
                return True, None

            # Giữ chỉ các cột cần thiết nếu có
            keep_cols = [c for c in ['time','open','close','volume'] if c in df.columns]
            if keep_cols:
                df = df[keep_cols]

            df.to_excel(out_path, index=False)
            print(f"✅ Saved {symbol} -> {out_path}")
            return True, out_path

        except SystemExit as se:
            # Bắt SystemExit => thư viện muốn terminate; thay vào đó parse thông điệp và sleep
            msg = str(se)
            wait_sec = parse_wait_seconds_from_msg(msg)
            if wait_sec:
                print(f"⏳ SystemExit caught for {symbol}. Server asked to wait {wait_sec}s. Sleeping...")
                time.sleep(wait_sec + 1)
            else:
                # fallback backoff
                print(f"⏳ SystemExit caught for {symbol}. No wait parsed. Backoff {backoff:.1f}s (attempt {attempt}/{max_attempts})")
                time.sleep(backoff)
                backoff *= 2

        except Exception as e:
            # Bắt các Exception khác (requests, ValueError, RateLimit trong message)
            msg = str(e)
            wait_sec = parse_wait_seconds_from_msg(msg)
            if wait_sec:
                print(f"⏳ Rate-limit message for {symbol}: wait {wait_sec}s. Sleeping...")
                time.sleep(wait_sec + 1)
            elif '429' in msg or 'rate limit' in msg.lower() or 'too many requests' in msg.lower():
                print(f"⏳ Rate-limit detected for {symbol}. Backoff {backoff:.1f}s (attempt {attempt}/{max_attempts})")
                time.sleep(backoff)
                backoff *= 2
            else:
                print(f"⚠️ Error for {symbol} (attempt {attempt}/{max_attempts}): {msg}")
                # những lỗi không phải rate limit: vẫn thử vài lần
                time.sleep(backoff)
                backoff *= 2

    print(f"❌ Give up {symbol} after {max_attempts} attempts.")
    return False, None

# ---- Orchestrator chính (resume-friendly) ----
progress = load_progress(progress_file)
if not progress:
    progress = {"done": [], "failed": []}

for sym in tqdm(vn30_symbols, desc="VN30 all"):
    if sym in progress.get("done", []):
        continue  # đã xong trước đó

    ok, path = fetch_symbol_to_file(sym)
    if ok:
        progress.setdefault("done", []).append(sym)
    else:
        progress.setdefault("failed", []).append(sym)

    save_progress(progress_file, progress)  # lưu progress liên tục

print("Finished. Done:", len(progress.get("done", [])), "Failed:", len(progress.get("failed", [])))
print("Files saved to:", save_folder)

In [3]:
# Các mã bị loại
from vnstock import Quote
import os, time, json, re, sys
from tqdm import tqdm
import pandas as pd

# ---- Cấu hình ----
save_folder = os.path.join("docs", "Elimiated_VN30")
os.makedirs(save_folder, exist_ok=True)

progress_file = os.path.join(save_folder, "progress.json")
start_date = '2020-01-01'
end_date   = '2025-11-10'
source     = 'TCBS'   # có thể đổi thành 'VCI' nếu muốn thử nguồn khác
interval   = '1D'

vn30_symbols = [
    'BVH', 'EIB', 'KDH', 'NVL', 'PDR', 'PNJ', 'REE', 'ROS', 'SBT', 'TCH'
]

# ---- Helpers ----
def load_progress(fn):
    if os.path.exists(fn):
        try:
            return json.load(open(fn, 'r'))
        except:
            return {}
    return {}

def save_progress(fn, data):
    with open(fn, 'w') as f:
        json.dump(data, f, indent=2)

def parse_wait_seconds_from_msg(msg):
    if not msg:
        return None
    # tìm pattern tiếng việt "xx giây"
    m = re.search(r'(\d+)\s*giây', msg, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    # tìm pattern tiếng anh "xx seconds" hoặc "retry after xx"
    m2 = re.search(r'(?:(\d+)\s*seconds)|(?:retry after[: ]?(\d+))', msg, flags=re.IGNORECASE)
    if m2:
        for g in m2.groups():
            if g and g.isdigit():
                return int(g)
    return None

# ---- Hàm tải với bắt SystemExit ----
def fetch_symbol_to_file(symbol,
                         start=start_date,
                         end=end_date,
                         source=source,
                         interval=interval,
                         out_folder=save_folder,
                         max_attempts=8,
                         initial_backoff=2.0):
    out_path = os.path.join(out_folder, f"{symbol}_{start.replace('-','')}_{end.replace('-','')}.xlsx")
    if os.path.exists(out_path):
        return True, out_path  # đã có, bỏ qua

    attempt = 0
    backoff = initial_backoff

    while attempt < max_attempts:
        attempt += 1
        try:
            # Thực hiện call
            quote = Quote(symbol=symbol, source=source)
            df = quote.history(symbol=symbol, start=start, end=end, interval=interval)

            # Nếu trả về None hoặc empty -> coi như valid nhưng báo warning
            if df is None or df.empty:
                print(f"⚠️ {symbol}: no data returned. Skipping save.")
                return True, None

            # Giữ chỉ các cột cần thiết nếu có
            keep_cols = [c for c in ['time','open','close','volume'] if c in df.columns]
            if keep_cols:
                df = df[keep_cols]

            df.to_excel(out_path, index=False)
            print(f"✅ Saved {symbol} -> {out_path}")
            return True, out_path

        except SystemExit as se:
            # Bắt SystemExit => thư viện muốn terminate; thay vào đó parse thông điệp và sleep
            msg = str(se)
            wait_sec = parse_wait_seconds_from_msg(msg)
            if wait_sec:
                print(f"⏳ SystemExit caught for {symbol}. Server asked to wait {wait_sec}s. Sleeping...")
                time.sleep(wait_sec + 1)
            else:
                # fallback backoff
                print(f"⏳ SystemExit caught for {symbol}. No wait parsed. Backoff {backoff:.1f}s (attempt {attempt}/{max_attempts})")
                time.sleep(backoff)
                backoff *= 2

        except Exception as e:
            # Bắt các Exception khác (requests, ValueError, RateLimit trong message)
            msg = str(e)
            wait_sec = parse_wait_seconds_from_msg(msg)
            if wait_sec:
                print(f"⏳ Rate-limit message for {symbol}: wait {wait_sec}s. Sleeping...")
                time.sleep(wait_sec + 1)
            elif '429' in msg or 'rate limit' in msg.lower() or 'too many requests' in msg.lower():
                print(f"⏳ Rate-limit detected for {symbol}. Backoff {backoff:.1f}s (attempt {attempt}/{max_attempts})")
                time.sleep(backoff)
                backoff *= 2
            else:
                print(f"⚠️ Error for {symbol} (attempt {attempt}/{max_attempts}): {msg}")
                # những lỗi không phải rate limit: vẫn thử vài lần
                time.sleep(backoff)
                backoff *= 2

    print(f"❌ Give up {symbol} after {max_attempts} attempts.")
    return False, None

# ---- Orchestrator chính (resume-friendly) ----
progress = load_progress(progress_file)
if not progress:
    progress = {"done": [], "failed": []}

for sym in tqdm(vn30_symbols, desc="VN30 all"):
    if sym in progress.get("done", []):
        continue  # đã xong trước đó

    ok, path = fetch_symbol_to_file(sym)
    if ok:
        progress.setdefault("done", []).append(sym)
    else:
        progress.setdefault("failed", []).append(sym)

    save_progress(progress_file, progress)  # lưu progress liên tục

print("Finished. Done:", len(progress.get("done", [])), "Failed:", len(progress.get("failed", [])))
print("Files saved to:", save_folder)

VN30 all: 100%|██████████| 20/20 [00:00<00:00, 227333.55it/s]

Finished. Done: 20 Failed: 0
Files saved to: docs/Elimiated_VN30
